# Phoneme to Text Converter by Fine-Tuning "T5-Small" LLM
## Training Data for Phoneme-to-Text LLM

I am using the pickle file to extract training data to make our own phoneme to NL decoder since we want to see what our decoded text looks like. This is going to help us qualitatively evaluate our models, and is also a cool exercise since we are interested in LLM fine-tuning as an extension of our work we've already done on the GRU baseline model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Fall 2025 UCLA/ECE_243A/Final Project/neural_seq_decoder/scripts

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Fall 2025 UCLA/ECE_243A/Final Project/neural_seq_decoder/scripts


In [ ]:
import pickle

# Path for pickle file which contains the phonemes and possibly text

path = '/content/drive/MyDrive/Fall 2025 UCLA/ECE_243A/Final Project/processed_data/ptDecoder_ctc.pkl'

with open(path, 'rb') as f:
    all_datasets = pickle.load(f)

# Print the data structure of the pickle file

print("\n Keys of pickle file")
print(list(all_datasets.keys()))

# Get the first datapoint from the train key

session_data = all_datasets['train'][0]

print("\n Keys inside the train key")
print(session_data.keys())


# Ground Truth Text
sample_text = session_data['transcriptions'][0]

# Ground Truth Phonemes
sample_phonemes = session_data['phonemes'][0]

print("\n Single sentence phoneme snippet")
print(f"1. Raw Transcription (Text): {sample_text}")
print(f"2. Phoneme Sequence: {sample_phonemes}")


 Keys of pickle file
['train', 'test', 'competition']

 Keys inside the train key
dict_keys(['sentenceDat', 'transcriptions', 'phonemes', 'timeSeriesLens', 'phoneLens', 'phonePerTime'])

 Single sentence phoneme snippet
1. Raw Transcription (Text): Nuclear rockets can destroy airfields with ease.
2. Phoneme Sequence: [23 34 20 21 18 12 40 28  1 20  3 31 29 40 20  2 23 40  9 17 29 31 28 26
 40 11 28 14 18 21  9 38 40 36 17 10 40 18 38 40  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  

# Translating Index to Phoneme

(Just to make sure the indices actually correspond to the right phonemes)

In [ ]:
# Using the same dictionary provided in the data processing notebook in reverse

PHONE_DEF = [
    'AA', 'AE', 'AH', 'AO', 'AW',
    'AY', 'B',  'CH', 'D', 'DH',
    'EH', 'ER', 'EY', 'F', 'G',
    'HH', 'IH', 'IY', 'JH', 'K',
    'L', 'M', 'N', 'NG', 'OW',
    'OY', 'P', 'R', 'S', 'SH',
    'T', 'TH', 'UH', 'UW', 'V',
    'W', 'Y', 'Z', 'ZH'
]
PHONE_DEF_SIL = PHONE_DEF + ['SIL']

# Go through phoneme indices and display the corresponding phonemes:

phoneme_text = ""

for index in sample_phonemes:

  # Need to use -1 since the phonemes are indexed by 1 (and 0 is end token)

  if index == 0:
    break

  phoneme = PHONE_DEF_SIL[index-1]

  phoneme_text += f'{phoneme} '

print(phoneme_text.strip())

N UW K L IY ER SIL R AA K AH T S SIL K AE N SIL D IH S T R OY SIL EH R F IY L D Z SIL W IH DH SIL IY Z SIL


# Actually Creating the Training Data

## Making the functions to implement my test above

Now that we have shown how to extract the sentence text as well as the corresponding phonemes, we can create a dataset that allows us to train the LLM. I am targeting the "T5-small" LLM since it is robust but should also be simple enough to train.

### Phoneme Index to Phoneme Sequence Function

In [ ]:
# Function to process the phoneme indices in the dataset into actual sequences
# so that the T5 can actually do its job of seq2seq decoding

# idx: the indices used for training the GRU and also what's returned by it
# alph: dictionary containing the phoneme strings (same one given for data processing in the start of the project)

def id_to_seq(idx,alph):
  phon_seq = []

  for i in idx:
    if i == 0:
      break

    phoneme = alph[i-1]

    phon_seq.append(phoneme)

  return(" ".join(phon_seq))

# - - - Test the function and see if it gives the right output - - -

# Get the datapoint from the train key

print(f'Number of Sentence Groups in Training Set: {len(all_datasets['train'][0]['transcriptions'])}')

session_data = all_datasets['train'][0]

print("\n Keys inside the train key")
print(session_data.keys())

# Ground Truth Text
sample_text = session_data['transcriptions'][279]

# Ground Truth Phonemes
sample_phonemes = session_data['phonemes'][279]

print(f'\n Sample Text: {sample_text}')

ph_from_id_test = id_to_seq(sample_phonemes,PHONE_DEF_SIL)

print(f'\n Sample Phoneme: {ph_from_id_test}')

Number of Sentence Groups in Training Set: 280

 Keys inside the train key
dict_keys(['sentenceDat', 'transcriptions', 'phonemes', 'timeSeriesLens', 'phoneLens', 'phonePerTime'])

 Sample Text: The irate actor stomped away idiotically.

 Sample Phoneme: DH AH SIL AY R EY T SIL AE K T ER SIL S T AA M P T SIL AH W EY SIL IH D IY AA T IH K L IY SIL


### Assembling the full Dataset for Fine Tuning

In [ ]:
# These lists will store the phoneme-sentence pairs

phoneme_data = []
text_data = []

# Iterate through all of the sentences and make the dataset

# I am using all of the possible sentences including the competition set
# (this is not "cheating" because the goal is to make a decoder, so it has
# minimal ability to generate from memory, hopefully)

for data_loc in ['train','test']:
  # For each "group" of sentences in the training data, we iterate through all
  # sentence-phoneme pairs and add them to the dataset lists
  for group in range(len(all_datasets[data_loc])):
    data_group = all_datasets[data_loc][group]
    # Inside each group, we see the individual pairs and process them
    for pair in range(len(data_group['transcriptions'])):
      text = data_group['transcriptions'][pair]
      phon_seq = id_to_seq(data_group['phonemes'][pair],PHONE_DEF_SIL)

      # Now we append to the data arrays:

      phoneme_data.append(phon_seq)
      text_data.append(text)

# - - - Test the dataset to see if we did everything right - - -

print(phoneme_data[258])
print(text_data[258])


AH SIL T UW TH P EY S T SIL T UW B SIL SH UH D SIL B IY SIL S K W IY Z D SIL F R AH M SIL DH AH SIL B AA T AH M SIL
A toothpaste tube should be squeezed from the bottom.


## Installing Training Tools and Dataset

Dataset makes data compatible with huggingface (where I'm storing the model)


In [ ]:
!pip install datasets transformers sentencepiece accelerate -q

In [ ]:
# Using the Huggingface Dataset tools to make the dataset for training

from datasets import Dataset

dataset = Dataset.from_dict({
    "phonemes": phoneme_data,
    "text": text_data
})

# Test that the dataset looks good:

dataset

Dataset({
    features: ['phonemes', 'text'],
    num_rows: 9680
})

### Pre-Processing Training Data for T5 Use

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_ver = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_ver)
model = T5ForConditionalGeneration.from_pretrained(model_ver)

# Tokenizer needs uniform-length data
# Giving more leeway for phonemes since they are longer strings

max_phon_len = 256
max_text_len = 128

def pre_process(data):
  # T5 needs a text prompt, not feature columns

  # Here, I am just adding a little prefix that makes it clear to decoded the
  # phonemes given to match the target, which is the sentence text

  inputs = []

  for phon in data['phonemes']:
    inputs.append('Transcribe phonemes to standard English: '+ phon)

  targets = data['text']

  # Tokenizing the raw text and phoneme strings so they work with the T5 training loop

  T5_inputs = tokenizer(inputs, truncation=True, padding="max_length",max_length=max_phon_len)

  T5_labels = tokenizer(targets, truncation=True, padding="max_length",max_length=max_text_len)

  # Put the labels into the 'labels' section of the tokenized input so the data is all in one place

  T5_inputs['labels'] = T5_labels['input_ids']

  return T5_inputs

# - - - Creating the dataset ready for T5 fine tuning and testing - - -

processed_dataset = dataset.map(
    pre_process,
    batched=True,
    remove_columns=dataset.column_names
)

processed_dataset

# Split data into train and test sets to keep tabs on the progress

train_test_split = processed_dataset.train_test_split(test_size=0.1)

train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

print(f"Training Examples: {len(train_dataset)}")
print(f"Validation Examples: {len(eval_dataset)}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/9680 [00:00<?, ? examples/s]

Training Examples: 8712
Validation Examples: 968


# Train the T5 LLM to do Decoding

Now that we have a compatible dataset with tokenized phonemes and text, we can run a training loop using Huggingface tools, which makes things much simpler and also allows me to save the model to my huggingface repo for later use in testing on the GRU model that we trained in the project.

In [ ]:
import wandb

wandb.login()

if wandb.run:
    wandb.finish()
    print("Previous WandB run successfully finished.")
else:
    print("No active WandB run to finish.")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


eval/loss,█▄▂▁▁
eval/runtime,▅█▅▃▁
eval/samples_per_second,▄▁▄▆█
eval/steps_per_second,▄▁▄▆█
train/epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███▁
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇█████
train/grad_norm,█▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▂▃▄▅▇███▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▁▁▁▁▄
train/loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,0.13062
eval/runtime,3.7114


Previous WandB run successfully finished.


In [ ]:
from transformers import DataCollatorForSeq2Seq, TrainingArguments, Trainer
import numpy as np # Needed for array ops in evaluation

# Assuming 'tokenizer', 'model', 'train_dataset', and 'eval_dataset' are defined.

data_collator = DataCollatorForSeq2Seq(
    tokenizer = tokenizer,
    model = model,
    padding = "longest"
)

training_args = TrainingArguments(

    output_dir = "./t5_phoneme_decoder",

    per_device_train_batch_size = 12,

    num_train_epochs = 3,

    learning_rate = 2e-4,
    weight_decay = 0.02,
    fp16 = True,
    logging_steps = 50,
    save_strategy = "epoch",
    lr_scheduler_type = "polynomial",
    warmup_ratio = 0.1,
    do_eval = True,
    eval_strategy = "epoch",
    load_best_model_at_end = True,
    report_to = "wandb",
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    data_collator = data_collator,
    tokenizer = tokenizer,
)

trainer.train()

/tmp/ipython-input-3829558042.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,0.130300,0.108370
2,0.108400,0.093889
3,0.099800,0.089176


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=2178, training_loss=0.11849615013413302, metrics={'train_runtime': 239.8034, 'train_samples_per_second': 108.989, 'train_steps_per_second': 9.082, 'total_flos': 1768646661636096.0, 'train_loss': 0.11849615013413302, 'epoch': 3.0})

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

repo_name = "BCI_proj_phoneme_decoder"
trainer.push_to_hub(repo_name)

print(f"Model saved to Hugging Face")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...neme_decoder/spiece.model: 100%|##########|  792kB /  792kB            

  ...decoder/model.safetensors:   0%|          |  552kB /  242MB            

  ...decoder/training_args.bin:   9%|9         |   551B / 5.84kB            

Model saved to Hugging Face


# Run Below to Test the Model

In [1]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 108.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Fall 2025 UCLA/ECE_243A/Final Project/neural_seq_decoder/scripts

Mounted at /content/drive
/content/drive/MyDrive/Fall 2025 UCLA/ECE_243A/Final Project/neural_seq_decoder/scripts


# Test Code
## Comparing Results of Shuffled and Unshuffled Training Data

In [13]:
import torch
import numpy as np
import pickle
import random
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed
from jiwer import wer
import string

# Start with our decoding dictionary to convert from indices to phonemes

PHONE_DEF = [
    'AA', 'AE', 'AH', 'AO', 'AW',
    'AY', 'B',  'CH', 'D', 'DH',
    'EH', 'ER', 'EY', 'F', 'G',
    'HH', 'IH', 'IY', 'JH', 'K',
    'L', 'M', 'N', 'NG', 'OW',
    'OY', 'P', 'R', 'S', 'SH',
    'T', 'TH', 'UH', 'UW', 'V',
    'W', 'Y', 'Z', 'ZH'
]
PHONE_DEF_SIL = PHONE_DEF + ['SIL']

# Same as before, we convert the indices to a phoneme sequence

def index_to_phoneme_sequence(idx_list):
    phoneme_text = ""
    for index in idx_list:
        if index == 0:
            break

        phoneme = PHONE_DEF_SIL[index - 1]
        phoneme_text += f'{phoneme} '

    return phoneme_text.strip()

# Function for shuffling the input sequences to test robustness

def shuffle_phoneme_sequence(phoneme_sequence, ground_truth):

    # Split phonemes up by sil character and text by spaces
    phoneme_groups = phoneme_sequence.split(' SIL ')
    text_words = ground_truth.split(' ')

    # put em in a list and shuffle at the same time
    paired_sequence = list(zip(phoneme_groups, text_words))
    random.shuffle(paired_sequence)

    # Get the phonemes and text back out of the list

    shuffled_phoneme_groups, shuffled_text_words = zip(*paired_sequence)

    # Reconstruct the shuffled phonemes and text with SIL and space

    shuffled_phoneme_sequence = ' SIL '.join(shuffled_phoneme_groups)

    shuffled_ground_truth_text = ' '.join(shuffled_text_words)

    return shuffled_phoneme_sequence, shuffled_ground_truth_text

# Now we need to perform inference on the index list to see the result of our model

def decode_from_indices(
    phoneme_sequence: str,
    hf_tokenizer: AutoTokenizer,
    hf_model: AutoModelForSeq2SeqLM,
    device: str) -> str:

    # Add the same prompt to the beginning of phonemes for T5 decoding
    input_text = "Transcribe phonemes to standard English: " + phoneme_sequence

    # Tokenize the conditioned input string
    current_input_ids = hf_tokenizer(
        input_text,
        return_tensors="pt",
        padding='max_length',
        max_length=256
    ).input_ids.to(device)

    # Generate the output sequence

    predicted_ids = hf_model.generate(

        # Had to mess with these params quite a lot to get an accurate output

        current_input_ids,
        max_length=128,
        early_stopping=True,
        #repetition_penalty=1.5,
        num_beams=5,
        do_sample=False,
        #length_penalty=0.2,
    )

    # Decode the predicted tokens to string format

    predicted_text = hf_tokenizer.decode(
        predicted_ids.squeeze(),
        skip_special_tokens=True
    )

    return predicted_text

# Get rid of capitals and punctuation to make the WER more accurate

def clean_text_for_wer(text):

    text = text.lower()

    text = text.translate(str.maketrans('', '', string.punctuation))

    return text

# Implement the complete decoding block which combines everything and calls the model

def full_decoding_pipeline(num_samples=20):

    MODEL_REPO_ID = "tonykorol/t5_phoneme_decoder"
    DATA_PATH = '/content/drive/MyDrive/Fall 2025 UCLA/ECE_243A/Final Project/processed_data/ptDecoder_ctc.pkl'

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load the model from my huggingface repo

    hf_tokenizer = AutoTokenizer.from_pretrained("t5-small")
    hf_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_REPO_ID)
    hf_model.to(device)

    # Load the data to test the decoding with:

    with open(DATA_PATH, 'rb') as f:
        all_datasets = pickle.load(f)

    test_dataset = all_datasets['test']
    test_dataset.extend(all_datasets['train'])

    # Choose random data index to decode

    dataset_size = len(test_dataset)
    random_indices = random.sample(range(dataset_size), num_samples)

    unshuffled_wers = []
    shuffled_wers = []

    # Iterate over the randomly selected samples
    for i, idx in enumerate(random_indices):
        sample = test_dataset[idx]
        raw_phoneme_indices = np.array(sample['phonemes']).flatten().tolist()
        ground_truth_text = sample['transcriptions'][0]

        # Create the unshuffled phoneme sequence from the index list given
        unshuffled_phoneme_seq = index_to_phoneme_sequence(raw_phoneme_indices)

        # Unshuffled decoding process

        unshuffled_decoded_text = decode_from_indices(
            phoneme_sequence=unshuffled_phoneme_seq,
            hf_tokenizer=hf_tokenizer,
            hf_model=hf_model,
            device=device
        )

        # WER for unshuffled text (With text normalization
        # because I noticed that it was punishing punctuation and case)

        unshuffled_wer = wer(
            reference=clean_text_for_wer(ground_truth_text),
            hypothesis=clean_text_for_wer(unshuffled_decoded_text)
        )
        unshuffled_wers.append(unshuffled_wer)

        # Create shuffled inputs

        shuffled_phoneme_seq, shuffled_ground_truth_text = shuffle_phoneme_sequence(
            phoneme_sequence=unshuffled_phoneme_seq,
            ground_truth=ground_truth_text
        )

        # Decode the shuffled examples

        shuffled_decoded_text = decode_from_indices(
            phoneme_sequence=shuffled_phoneme_seq,
            hf_tokenizer=hf_tokenizer,
            hf_model=hf_model,
            device=device
        )
        # Calculate WER for the shuffled examples

        shuffled_wer = wer(
            reference=clean_text_for_wer(shuffled_ground_truth_text),
            hypothesis=clean_text_for_wer(shuffled_decoded_text)
        )
        shuffled_wers.append(shuffled_wer)


        # Display 5 examples to take a look at

        if i < 5:
            # Unshuffled

            print(f"{i + 1} - Not Shuffled")
            print(f"Input Sequence:         Transcribe phonemes to standard English: {unshuffled_phoneme_seq}")
            print(f"Ground Truth Text:       {ground_truth_text}")
            print(f"Modeled Decoded Text:     {unshuffled_decoded_text}")
            print(f"WER:   {unshuffled_wer:.4f}\n")

            # shuffled results
            print(f"{i + 1} - Shuffled")
            print(f"Input Sequence (Shuffled): Transcribe phonemes to standard English: {shuffled_phoneme_seq}")
            print(f"Ground Truth (Shuffled):   {shuffled_ground_truth_text}")
            print(f"Modeled Decoded Text:     {shuffled_decoded_text}")
            print(f"WER):   {shuffled_wer:.4f}\n\n")


    # Calculate average wer and compare between shuffled and non shuffled
    avg_unshuffled_wer = np.mean(unshuffled_wers)
    avg_shuffled_wer = np.mean(shuffled_wers)


    print(f"WER Results Over {num_samples} Samples")
    print(f"Average Unshuffled WER: {avg_unshuffled_wer:.2f}")
    print(f"Average Shuffled WER: {avg_shuffled_wer:.2f}")

# Run the pipeline
full_decoding_pipeline(num_samples=30)

1 - Not Shuffled
Input Sequence:         Transcribe phonemes to standard English: IH T SIL D IH P EH N D Z SIL AA N SIL DH AH SIL P ER S AH N SIL
Ground Truth Text:       It depends on the person.
Modeled Decoded Text:     It depends on the person.
WER:   0.0000

1 - Shuffled
Input Sequence (Shuffled): Transcribe phonemes to standard English: DH AH SIL P ER S AH N SIL SIL AA N SIL D IH P EH N D Z SIL IH T
Ground Truth (Shuffled):   the person. on depends It
Modeled Decoded Text:     The person on duty is it.
WER):   0.4000


2 - Not Shuffled
Input Sequence:         Transcribe phonemes to standard English: DH EY SIL D OW N T SIL D UW SIL DH AE T SIL
Ground Truth Text:       They don't do that.
Modeled Decoded Text:     They don't do that.
WER:   0.0000

2 - Shuffled
Input Sequence (Shuffled): Transcribe phonemes to standard English: D OW N T SIL DH AE T SIL SIL DH EY SIL D UW
Ground Truth (Shuffled):   don't that. They do
Modeled Decoded Text:     Don't that they do.
WER):   0.0000


3 